In [ ]:
from pathlib import Path
from metasmith.python_api import Agent, Source, Std, DataInstanceLibrary, TransformInstanceLibrary, Endpoint

dtypes, containers, transforms = Std()

In [ ]:
path_to_agent_home = Path("./cache/local_home").resolve()
smith = Agent(
    home = Source.FromLocal(path_to_agent_home),
)
smith.Deploy()

In [ ]:
from metasmith.coms.ipc import LiveShell
from local.constants import WORKSPACE_ROOT

home = f"{WORKSPACE_ROOT}/main/tests/cache/local_home"

with LiveShell() as shell:
    shell.RegisterOnOut(lambda x: print(x))
    shell.RegisterOnErr(lambda x: print(f"E: {x}"))
    # shell.Exec(f"rsync -ac --progress --mkpath {WORKSPACE_ROOT}/metasmith.sif {home}/metasmith.sif")
    shell.Exec(f"rsync -acu --progress --mkpath {WORKSPACE_ROOT}/main/relay_agent/dist/relay {home}/relay/msm_relay")
    shell.Exec(f"rsync -acu --progress --mkpath --exclude=__pycache__ {WORKSPACE_ROOT}/src/metasmith/ {home}/dev/metasmith")
    shell.Exec(f"rsync -acu --progress --mkpath --exclude=__pycache__ {WORKSPACE_ROOT}/src/metasmith/nextflow_config {home}/lib/")


In [ ]:
# # generate transform lib template
# transforms = TransformInstanceLibrary("./exec_with_container.xgdb")
# transforms.AddStub("mre.py")
# transforms.Save()

transforms = TransformInstanceLibrary.Load("./exec_with_container.xgdb")
for _path, _, tr in transforms.IterateTransforms():
    print(tr.name)
    for p in tr.model.requires:
        print(" ", p)
    print("->")
    for p in tr.model.produces:
        print(" ", p)

In [ ]:
task = smith.GenerateWorkflow(
    given=[containers],
    transforms=[transforms],
    targets=[dtypes.types["assembly"]]
)
for x in task.plan.steps:
    print(x.transform.name)

In [ ]:
smith.StageWorkflow(task, on_exist="clear")
smith.RunWorkflow(task)

In [ ]:
with LiveShell() as shell:
    r = shell.Exec("echo asdf", history=True)
    print(r.out)